# Concours MTH3302

In [148]:
using CSV
using DataFrames
using Gadfly
using HypothesisTests
using LinearAlgebra
using Statistics
using StatsPlots
using Combinatorics
using StatsModels
using CategoricalArrays
using StatsBase
using Random

include("Jeremie_Utils.jl")

bayesian_prediction (generic function with 3 methods)

In [ ]:
train = CSV.read("train.csv", DataFrame, decimal=',')
test = CSV.read("test.csv", DataFrame, decimal=',');

# Partie 1
## Régressions linéaires simples

In [ ]:
y = train.consommation
n = length(y)

In [152]:
function compute_residuals(model, y)
    ŷ = StatsModels.predict(model)
    res = (y - ŷ) / std(ŷ)

    return res
end

function plot_explanatory_variable(model, data, xlabel)
    predictions = StatsModels.predict(model)

    Gadfly.plot(
        x = data.x, 
        y = data.y,
        layer(
            x = data.x,
            y = predictions,
            Geom.line,
            Theme(default_color="red"),
        ),
        Geom.point,
        Guide.xlabel(xlabel), 
        Guide.ylabel("Consommation d'essence (L/100km)", orientation=:vertical),
    )
end

function qqplot_test(model, data)
    errors = compute_residuals(model, data.y)

    qqnorm(errors, qqline = :R)
end

function shapiro_wilk_test(model, data)
    errors = compute_residuals(model, data.y)
    p = pvalue(ShapiroWilkTest(errors))

    if p > 0.05
        println("$p > 0.05 -> On accepte l'hypothèse que les données proviennent d'une distribution normale")
    else 
        println("$p ≤ 0.05 -> On rejette l'hypothèse que les données proviennent d'une distribution normale")
    end
end

function residuals_vs_fitted_values_plot_test(model, data)
    errors = compute_residuals(model, data.y)
    fitted_values = fitted(model)

    Gadfly.plot(
        layer(x=fitted_values, y=errors, Geom.point),
        Guide.xlabel("Valeurs prédites"),
        Guide.ylabel("Résidus"),
    )
end

function residuals_vs_observation_order_plot_test(model, data)
    errors = compute_residuals(model, data.y)

    Gadfly.plot(
        layer(x=1:length(errors), y=errors, Geom.point),
        Guide.xlabel("Index"),
        Guide.ylabel("Résidus"),
    )
end

function get_data_set(seed, features, preprocess::Function = data -> nothing)
    Random.seed!(seed)
    data = CSV.read("train.csv", DataFrame, decimal=',')

    preprocess(data)

    train_id = sample(1:nrow(data), round(Int, .8*nrow(data)), ordered=true, replace=false)
    valid_id = setdiff(1:nrow(data), train_id)

    train = data[train_id,:]
    train = remove_outliers(train, features, :consommation)
    valid = data[valid_id,:]

    return train, valid
end

function align_categorical_features(train, valid, features)
    data = vcat(train, valid)

    categorical_features = filter(x -> eltype(data[:, x]) <: AbstractString || eltype(data[:, x]) <: CategoricalValue, features)
    for feature in categorical_features
        train_values = unique(train[!, feature])
        valid_values = unique(valid[!, feature])

        train_indices_to_remove = findall(x -> !(x in valid_values), train[!, feature])
        train = train[setdiff(1:nrow(train), train_indices_to_remove), :]

        valid_indices_to_remove = findall(x -> !(x in train_values), valid[!, feature])
        valid = valid[setdiff(1:nrow(valid), valid_indices_to_remove), :]
    end

    return train, valid
end

function update_features(data, features) 
    return vec(reduce(vcat, [
        filter(name -> startswith(name, string(feature)), names(data))
        for feature in features
    ]))
end

update_features (generic function with 1 method)

## 1.1 Analyse de la variable _nombre_cylindres_

In [ ]:
data = DataFrame(y = train.consommation, x = log.(train.nombre_cylindres))
model = create_model(data, [:x])

**Vérification de l'hypothèse de linéarité**

In [ ]:
plot_explanatory_variable(model, data, "Nombre de cylindres")

**Vérification de l'hypothèse de normalité des erreurs**

In [ ]:
qqplot_test(model, data)

In [ ]:
shapiro_wilk_test(model, data)

**Vérification de l'hypothèse d'homosédasticité des erreurs**

In [ ]:
residuals_vs_fitted_values_plot_test(model, data)

**Vérification de l'hypothèse d'indépendance des erreurs**

In [ ]:
residuals_vs_observation_order_plot_test(model, data)

In [ ]:
bic(model)

## 1.2 Analyse de la variable _type_

In [ ]:
data = DataFrame(y = train.consommation, x = train.type)
model = create_model(data, [:x])

**Vérification de l'hypothèse de linéarité**

In [ ]:
plot_explanatory_variable(model, data, "Type")

**Vérification de l'hypothèse de normalité des erreurs**

In [ ]:
qqplot_test(model, data)

In [ ]:
shapiro_wilk_test(model, data)

**Vérification de l'hypothèse d'homosédasticité des erreurs**

In [ ]:
residuals_vs_fitted_values_plot_test(model, data)

**Vérification de l'hypothèse d'indépendance des erreurs**

In [ ]:
residuals_vs_observation_order_plot_test(model, data)

In [ ]:
r2(model)

## 1.3 Analyse de la variable _cylindree_

In [ ]:
data = DataFrame(y = train.consommation, x = log.(train.cylindree))
model = create_model(data, [:x])

# data = DataFrame(y = train.consommation, x = train.cylindree, x² = train.cylindree .^ 2)
# model = create_model(data, [:x, :x²])

**Vérification de l'hypothèse de linéarité**

In [ ]:
plot_explanatory_variable(model, data, "Cylindrée")

**Vérification de l'hypothèse de normalité des erreurs**

In [ ]:
qqplot_test(model, data)

In [ ]:
shapiro_wilk_test(model, data)

**Vérification de l'hypothèse d'homosédasticité des erreurs**

In [ ]:
residuals_vs_fitted_values_plot_test(model, data)

**Vérification de l'hypothèse d'indépendance des erreurs**

In [ ]:
residuals_vs_observation_order_plot_test(model, data)

In [ ]:
r2(model)

## 1.4 Analyse de la variable _transmission_

In [ ]:
x = train.transmission
data = DataFrame(y = train.consommation, x = x)
data = remove_outliers(data)
model = create_model(data, [:x])

**Vérification de l'hypothèse de linéarité**

In [ ]:
plot_explanatory_variable(model, data, "Transmission")

**Vérification de l'hypothèse de normalité des erreurs**

In [ ]:
qqplot_test(model, data)

In [ ]:
shapiro_wilk_test(model, data)

**Vérification de l'hypothèse d'homosédasticité des erreurs**

In [ ]:
residuals_vs_fitted_values_plot_test(model, data)

**Vérification de l'hypothèse d'indépendance des erreurs**

In [ ]:
residuals_vs_observation_order_plot_test(model, data)

In [ ]:
r2(model)

## 1.5 Analyse de la variable _boite_

In [ ]:
data = DataFrame(y = train.consommation, x = train.boite)
data = remove_outliers(data)
model = create_model(data, [:x])

**Vérification de l'hypothèse de linéarité**

In [ ]:
plot_explanatory_variable(model, data, "Cylindrée")

**Vérification de l'hypothèse de normalité des erreurs**

In [ ]:
qqplot_test(model, data)

In [ ]:
shapiro_wilk_test(model, data)

**Vérification de l'hypothèse d'homosédasticité des erreurs**

In [ ]:
residuals_vs_fitted_values_plot_test(model, data)

**Vérification de l'hypothèse d'indépendance des erreurs**

In [ ]:
residuals_vs_observation_order_plot_test(model, data)

In [ ]:
r2(model)

## 1.6 Analyse de la variable _annee_

In [ ]:
data = DataFrame(y = train.consommation, x = train.annee)
coerce!(data, :x => Multiclass)
model = create_model(data, [:x])

In [ ]:
plot_explanatory_variable(model, data, "Année")

In [ ]:
r2(model)

# Partie 2 
## Régression linéaire multiple

In [ ]:
seed = 9514

features = [:annee, :type, :nombre_cylindres, :cylindree, :transmission, :boite]
train, valid = get_data_set(seed, features)
model = create_model(train, features, :consommation)

ŷ = float.(StatsModels.predict(model, valid))
y_valid = valid[:, :consommation]

rms(ŷ, y_valid)

In [ ]:
features = [:annee, :type, :nombre_cylindres, :cylindree, :transmission]
train, valid = get_data_set(seed, features)

model = create_model(train, features, :consommation)

ŷ = float.(StatsModels.predict(model, valid))
y_valid = valid[:, :consommation]

rms(ŷ, y_valid)

In [ ]:
features = [:annee, :type, :nombre_cylindres, :log_cylindree, :transmission, :boite]
train, valid = get_data_set(seed, features, data -> begin
    data.log_cylindree = log.(data.cylindree)
end)

model = create_model(train, features, :consommation)

ŷ = float.(StatsModels.predict(model, valid))
y_valid = valid[:, :consommation]

rms(ŷ, y_valid)

In [ ]:
features = [:annee, :type, :log_nombre_cylindres, :cylindree, :transmission, :boite]
train, valid = get_data_set(seed, features, data -> begin
    data.log_nombre_cylindres = log.(data.nombre_cylindres)
end)

model = create_model(train, features, :consommation)

ŷ = float.(StatsModels.predict(model, valid))
y_valid = valid[:, :consommation]

rms(ŷ, y_valid)

In [ ]:
features = [:annee, :type, :nombre_cylindres, :cylindree, :transmission, :boite]
train, valid = get_data_set(seed, features, data -> begin
    coerce!(data, :annee => Multiclass)
end)

model = create_model(train, features, :consommation)

ŷ = float.(StatsModels.predict(model, valid))
y_valid = valid[:, :consommation]

rms(ŷ, y_valid)

In [174]:
features = [:annee, :type, :nombre_cylindres, :cylindree, :transmission, :boite]
train, valid = get_data_set(seed, features, data -> begin
    data[!, :annee] = string.(data[!, :annee])
    data[!, :nombre_cylindres] = float.(data[!, :nombre_cylindres])
end)
train, valid = align_categorical_features(train, valid, features)

train = one_hot_encode(train, features)
valid = one_hot_encode(valid, features)
features = update_features(train, features)

X_train = train[!, features]
y_train = train[!, :consommation]

X_valid = valid[!, features]

mach = linear_regression(X_train, y_train)

ŷ = MLJ.predict(mach, X_valid)
y_valid = valid[:, :consommation]

rms(ŷ, y_valid)

0.8773887659629036

In [175]:
include("Jeremie_Utils.jl")
features = [:annee, :type, :nombre_cylindres, :cylindree, :transmission, :boite]
train, valid = get_data_set(seed, features, data -> begin
    data[!, :annee] = string.(data[!, :annee])
    data[!, :nombre_cylindres] = float.(data[!, :nombre_cylindres])
end)
train, valid = align_categorical_features(train, valid, features)

train = one_hot_encode(train, features)
valid = one_hot_encode(valid, features)

features = update_features(train, features)

X_train = train[!, features]
y_train = train[!, :consommation]

X_valid = valid[!, features]

mach = ridge_regression_cv(X_train, y_train)

ŷ = MLJ.predict(mach, X_valid)
y_valid = valid[:, :consommation]

rms(ŷ, y_valid)

0.8558274000977868

In [161]:
features = [:annee, :type, :nombre_cylindres, :cylindree, :transmission, :boite]
train, valid = get_data_set(seed, features, data -> begin
    data[!, :annee] = string.(data[!, :annee])
    data[!, :nombre_cylindres] = float.(data[!, :nombre_cylindres])
end)
train, valid = align_categorical_features(train, valid, features)

train = one_hot_encode(train, features)
valid = one_hot_encode(valid, features)

features = update_features(train, features)

X_train = train[!, features]
y_train = train[!, :consommation]

X_valid = valid[!, features]

mach = lasso_regression_cv(X_train, y_train)

ŷ = MLJ.predict(mach, X_valid)
y_valid = valid[:, :consommation]

rms(ŷ, y_valid)

┌ Warning: Proximal GD did not converge in 1000 iterations.
└ @ MLJLinearModels /Users/jeremiebolduc/.julia/packages/MLJLinearModels/TXgHx/src/fit/proxgrad.jl:64
┌ Warning: Proximal GD did not converge in 1000 iterations.
└ @ MLJLinearModels /Users/jeremiebolduc/.julia/packages/MLJLinearModels/TXgHx/src/fit/proxgrad.jl:64
┌ Warning: Proximal GD did not converge in 1000 iterations.
└ @ MLJLinearModels /Users/jeremiebolduc/.julia/packages/MLJLinearModels/TXgHx/src/fit/proxgrad.jl:64
┌ Warning: Proximal GD did not converge in 1000 iterations.
└ @ MLJLinearModels /Users/jeremiebolduc/.julia/packages/MLJLinearModels/TXgHx/src/fit/proxgrad.jl:64
┌ Warning: Proximal GD did not converge in 1000 iterations.
└ @ MLJLinearModels /Users/jeremiebolduc/.julia/packages/MLJLinearModels/TXgHx/src/fit/proxgrad.jl:64
┌ Warning: Proximal GD did not converge in 1000 iterations.
└ @ MLJLinearModels /Users/jeremiebolduc/.julia/packages/MLJLinearModels/TXgHx/src/fit/proxgrad.jl:64
┌ Warning: Proximal GD did n

0.8637095919518141

In [166]:
include("Jeremie_Utils.jl")
features = [:annee, :type, :nombre_cylindres, :cylindree, :transmission, :boite]
train, valid = get_data_set(seed, features, data -> begin
    data[!, :annee] = string.(data[!, :annee])
    data[!, :nombre_cylindres] = float.(data[!, :nombre_cylindres])
end)
train, valid = align_categorical_features(train, valid, features)

train = one_hot_encode(train, features)
valid = one_hot_encode(valid, features)

features = update_features(train, features)

X_train = train[!, features]
y_train = train[!, :consommation]

X_valid = valid[!, features]

mach = elastic_net_regression_cv(X_train, y_train)

ŷ = MLJ.predict(mach, X_valid)
y_valid = valid[:, :consommation]

rms(ŷ, y_valid)

┌ Warning: Proximal GD did not converge in 1000 iterations.
└ @ MLJLinearModels /Users/jeremiebolduc/.julia/packages/MLJLinearModels/TXgHx/src/fit/proxgrad.jl:64
┌ Warning: Proximal GD did not converge in 1000 iterations.
└ @ MLJLinearModels /Users/jeremiebolduc/.julia/packages/MLJLinearModels/TXgHx/src/fit/proxgrad.jl:64
┌ Warning: Proximal GD did not converge in 1000 iterations.
└ @ MLJLinearModels /Users/jeremiebolduc/.julia/packages/MLJLinearModels/TXgHx/src/fit/proxgrad.jl:64
┌ Warning: Proximal GD did not converge in 1000 iterations.
└ @ MLJLinearModels /Users/jeremiebolduc/.julia/packages/MLJLinearModels/TXgHx/src/fit/proxgrad.jl:64
┌ Warning: Proximal GD did not converge in 1000 iterations.
└ @ MLJLinearModels /Users/jeremiebolduc/.julia/packages/MLJLinearModels/TXgHx/src/fit/proxgrad.jl:64
┌ Warning: No appropriate stepsize found via backtracking; interrupting. The reason could be input data that is not standardized.
└ @ MLJLinearModels /Users/jeremiebolduc/.julia/packages/MLJ

0.8770668635489842

In [ ]:
include("Jeremie_Utils.jl")

features = [:annee, :type, :nombre_cylindres, :cylindree, :transmission, :boite]
train, valid = get_data_set(seed, features, data -> begin
    data[!, :annee] = string.(data[!, :annee])
    data[!, :nombre_cylindres] = float.(data[!, :nombre_cylindres])
end)

train = one_hot_encode(train, features)
valid = one_hot_encode(valid, features)

features = update_features(train, features)

X_train = Matrix(train[!, features])
y_train = train[!, :consommation]

X_valid = Matrix(valid[!, features])

model = bayesian_regression(X_train, y_train)
chain = sample(model, NUTS(), 500)

In [ ]:
include("Jeremie_Utils.jl")

ŷ = bayesian_prediction(chain, X_valid, mean)
y_valid = valid[:, :consommation]

rms(ŷ, y_valid)

In [ ]:
vif = 1 / (1 - r2(model))

In [ ]:
data = deepcopy(train)
feature_combinations = filter(x -> !isempty(x), [comb for i in 0:length(features) for comb in combinations(features, i)])
min_rmse = Inf
best_model = missing
best_features = missing

for feature_combination in feature_combinations
    mᵢ = create_model(data, feature_combination, :consommation)

    ŷ = StatsModels.predict(mᵢ, valid)
    y_valid = valid[:, :consommation]

    res = rms(ŷ, y_valid)
    if res < min_rmse
        min_rmse = res
        best_model = mᵢ
        best_features = feature_combination
    end
end

best_features

In [ ]:
id = 1:size(test, 1)

ŷ = MLJ.predict(mach, test)
# ŷ = StatsModels.predict(model, test)

# df_pred = DataFrame(id=id, consommation=ŷ)

# CSV.write("benchmark.csv", df_pred)

### Régression Ridge pour soumission

In [179]:
test = CSV.read("test.csv", DataFrame, decimal=',');
test[!, :annee] = string.(test[!, :annee])
test[!, :nombre_cylindres] = float.(test[!, :nombre_cylindres])

features = [:annee, :type, :nombre_cylindres, :cylindree, :transmission, :boite]
train, valid = get_data_set(seed, features, data -> begin
    data[!, :annee] = string.(data[!, :annee])
    data[!, :nombre_cylindres] = float.(data[!, :nombre_cylindres])
end)
train = vcat(train, valid)

X_train = train[!, features]
y_train = train[!, :consommation]

X_train, test = align_categorical_features(X_train, test, features)

X_train = one_hot_encode(X_train, features)
test = one_hot_encode(test, features)
features = update_features(X_train, features)

mach = ridge_regression_cv(X_train, y_train)

ŷ = MLJ.predict(mach, test)

df_pred = DataFrame(id=id, consommation=ŷ)
CSV.write("benchmark.csv", df_pred)

"benchmark.csv"